In [19]:
!C:\\Users\\teunv\\anaconda3\\envs\\ir_project\\python.exe -m pip install pyterrier-alpha

In [18]:
import sys
sys.executable

'C:\\Users\\teunv\\anaconda3\\envs\\ir_project\\python.exe'

# Experiment 1: Baseline SPLADE

In [1]:
import pyterrier as pt
import pyt_splade
import os
import torch
import warnings
import shutil

warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")

torch.manual_seed(26)  # for reproducibility

def custom_iter():
    for doc in dataset.get_corpus_iter():
        yield {
            'docno': doc['docno'],
            'text': (doc.get('title','') + ' ' + doc.get('body','')).strip()
        }

def detect_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"

device = detect_device()
print(f"Using device: {device}")

splade = pyt_splade.Splade(model="C:\splade_model", device=device, max_length=256)

# Load Robust04 dataset
dataset = pt.get_dataset("irds:disks45/nocr/trec-robust-2004")

# Set up indexer
index_dir = os.path.abspath("./robust04_splade_index")
if os.path.exists(index_dir):
    try:
        shutil.rmtree(index_dir)
    except OSError:
        raise RuntimeError("File Lock Error")

indexer = pt.IterDictIndexer(index_dir, pretokenised=True, verbose=True, overwrite=True)

# Create an indexing pipeline: encode documents with SPLADE, then index
indexing_pipeline = splade.doc_encoder(batch_size=64) >> indexer

# Build the index (this may take a while for large datasets)
index_ref = indexing_pipeline.index(custom_iter(), batch_size=64)

Using device: cuda


Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


disks45/nocr/trec-robust-2004 documents:   0%|          | 0/528155 [00:00<?, ?it/s]

In [14]:
from pyterrier.measures import *

splade_retr = splade.query_encoder() >> pt.terrier.Retriever(index_ref, wmodel="Tf")
topics = dataset.get_topics()
topics["query"] = topics["title"]
qrels = dataset.get_qrels()

results = pt.Experiment(
    [splade_retr],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG@10, Recall@100],
    names=["SPLADE Baseline"]
)

print(results)

There are multiple query fields available: ('title', 'description', 'narrative'). To use with pyterrier, provide variant or modify dataframe to add query column.
              name        AP     R@100   nDCG@10
0  SPLADE Baseline  0.225201  0.372996  0.454875


# Experiment 2: Improved SPLADE with Hybrid Retrieval

In [11]:
bm25_index_dir = os.path.abspath("./robust04_bm25_index")

if not os.path.exists(bm25_index_dir):
    bm25_indexer = pt.IterDictIndexer(bm25_index_dir)
    bm25_index_ref = bm25_indexer.index(custom_iter())
else:
    bm25_index_ref = pt.IndexFactory.of(bm25_index_dir)

disks45/nocr/trec-robust-2004 documents:   0%|          | 0/528155 [00:00<?, ?it/s]

20:48:16.393 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (FR940104-2-00103) - further warnings are suppressed
20:57:30.249 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 125 empty documents


In [13]:
splade_retr = splade.query_encoder() >> pt.terrier.Retriever(index_ref, wmodel="Tf")
bm25_retr = pt.terrier.Retriever(bm25_index_ref, wmodel="BM25")

# Hybrid Retrieval with different weights for bm25
hybrid_05 = splade_retr + (0.05 * bm25_retr)
hybrid_10 = splade_retr + (0.10 * bm25_retr)
hybrid_20 = splade_retr + (0.20 * bm25_retr)

In [15]:
from pyterrier.measures import *

topics = dataset.get_topics()
topics["query"] = topics["title"]
qrels = dataset.get_qrels()

experiment = pt.Experiment(
    [splade_retr, bm25_retr, hybrid_05, hybrid_10, hybrid_20],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG@10, Recall@100],
    names=["SPLADE Only", "BM25 Only", "Hybrid (w=0.05)", "Hybrid (w=0.1)", "Hybrid (w=0.2)"]
)

print(experiment)

There are multiple query fields available: ('title', 'description', 'narrative'). To use with pyterrier, provide variant or modify dataframe to add query column.
              name        AP     R@100   nDCG@10
0      SPLADE Only  0.225201  0.372996  0.454875
1        BM25 Only  0.236869  0.401218  0.424420
2  Hybrid (w=0.05)  0.231541  0.373555  0.454920
3   Hybrid (w=0.1)  0.231898  0.373763  0.455269
4   Hybrid (w=0.2)  0.232330  0.373926  0.456622


# Experiment 3: Reciprocal Rank Fusion

In [23]:
import pyterrier_alpha as pta

splade_res = splade_retr.transform(topics)
bm25_res = bm25_retr.transform(topics)

rrf = pta.fusion.rr_fusion(splade_res, bm25_res, k=60)

experiment = pt.Experiment(
    [splade_res, bm25_res, hybrid_20, rrf],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG@10, Recall@100],
    names=["SPLADE", "BM25", "Hybrid (w=0.2)", "RRF"]
)

print(experiment)

             name        AP     R@100   nDCG@10
0          SPLADE  0.225201  0.372996  0.454875
1            BM25  0.236869  0.401218  0.424420
2  Hybrid (w=0.2)  0.232330  0.373926  0.456622
3             RRF  0.259509  0.427527  0.465380


# Experiment 4: Hybrid Retrieval Weight Optimisation

In [24]:
import numpy as np
import pandas as pd
import pyterrier as pt
import pyterrier_alpha as pta

weights = np.arange(0.0, 1.1, 0.1)
results = []

for w in weights:
    hybrid = splade_retr + (w * bm25_retr)
    exp = pt.Experiment(
        [hybrid],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG@10, Recall@100],
        names=[f'Hybrid (w={w:.1f})']
    )
    results.append(exp)

hybrid_results = pd.concat(results)
print(hybrid_results)

             name        AP     R@100   nDCG@10
0  Hybrid (w=0.0)  0.230722  0.372996  0.454875
0  Hybrid (w=0.1)  0.231898  0.373763  0.455269
0  Hybrid (w=0.2)  0.232330  0.373926  0.456622
0  Hybrid (w=0.3)  0.233116  0.374879  0.457127
0  Hybrid (w=0.4)  0.233623  0.375548  0.457471
0  Hybrid (w=0.5)  0.234130  0.375900  0.458007
0  Hybrid (w=0.6)  0.234614  0.376678  0.459487
0  Hybrid (w=0.7)  0.235166  0.377217  0.460145
0  Hybrid (w=0.8)  0.235629  0.377815  0.460163
0  Hybrid (w=0.9)  0.236122  0.379031  0.460352
0  Hybrid (w=1.0)  0.236392  0.379696  0.460865


In [35]:
import numpy as np
import pandas as pd
import pyterrier as pt
import pyterrier_alpha as pta

weights = [1, 2, 3, 5, 8, 10, 15, 20]
results = []

for w in weights:
    hybrid = splade_retr + (w * bm25_retr)
    exp = pt.Experiment(
        [hybrid],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG@10, Recall@100],
        names=[f'Hybrid (w={w:.1f})']
    )
    results.append(exp)

hybrid_results = pd.concat(results)
print(hybrid_results)

              name        AP     R@100   nDCG@10
0   Hybrid (w=1.0)  0.236392  0.379696  0.460865
0   Hybrid (w=2.0)  0.240249  0.384782  0.465489
0   Hybrid (w=3.0)  0.242948  0.387588  0.470548
0   Hybrid (w=5.0)  0.247422  0.391799  0.477217
0   Hybrid (w=8.0)  0.249458  0.393393  0.479059
0  Hybrid (w=10.0)  0.249904  0.395853  0.481710
0  Hybrid (w=15.0)  0.252479  0.396115  0.486237
0  Hybrid (w=20.0)  0.255466  0.399816  0.486462


In [36]:
import numpy as np
import pandas as pd
import pyterrier as pt
import pyterrier_alpha as pta

weights = [20, 30, 40, 50, 75, 100]
results = []

for w in weights:
    hybrid = splade_retr + (w * bm25_retr)
    exp = pt.Experiment(
        [hybrid],
        topics,
        qrels,
        eval_metrics=[MAP, nDCG@10, Recall@100],
        names=[f'Hybrid (w={w:.1f})']
    )
    results.append(exp)

hybrid_results = pd.concat(results)
print(hybrid_results)

               name        AP     R@100   nDCG@10
0   Hybrid (w=20.0)  0.255466  0.399816  0.486462
0   Hybrid (w=30.0)  0.259681  0.409047  0.480513
0   Hybrid (w=40.0)  0.262010  0.411482  0.474168
0   Hybrid (w=50.0)  0.262278  0.414359  0.471348
0   Hybrid (w=75.0)  0.260799  0.416585  0.463245
0  Hybrid (w=100.0)  0.259823  0.416573  0.456647


# Experiment 5: Expansion Models

In [40]:
hybrid_20 = splade_retr + (20 * bm25_retr)

bm25_bo1 = (
    bm25_retr
    >> pt.text.get_text(dataset, ["title", "body"])
    
    >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
    
    >> pt.rewrite.Bo1QueryExpansion(bm25_index_ref, fb_docs=3, fb_terms=10)
    
    >> bm25_retr
)

upgraded_hybrid = splade_retr + (20.0 * bm25_bo1)

pt.Experiment(
    [splade_retr, hybrid_20, upgraded_hybrid], 
    topics,
    qrels,
    eval_metrics=[MAP, nDCG@10, Recall@100],
    names=["SPLADE", "Hybrid (w=20)", "Upgraded Hybrid (w=20 + Bo1)"]
)

[INFO] [starting] building docstore
docs_iter: 528155doc [06:43, 1310.32doc/s]
[INFO] [finished] docs_iter: [06:43] [528155doc] [1310.31doc/s]
[INFO] [finished] building docstore [06:43]


,name,AP,R@100,nDCG@10
0,SPLADE,0.225201,0.372996,0.454875
1,Hybrid (w=20),0.255466,0.399816,0.486462
2,Upgraded Hybrid (w=20 + Bo1),0.273956,0.423448,0.490863


In [41]:
hybrid_20 = splade_retr + (20 * bm25_retr)

bm25_rm3 = (
    bm25_retr
    >> pt.text.get_text(dataset, ["title", "body"])
    
    >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
    
    >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=10)
    
    >> bm25_retr
)

rm3_hybrid = splade_retr + (20.0 * bm25_rm3)

pt.Experiment(
    [rm3_hybrid],
    topics,
    qrels,
    eval_metrics=[MAP, nDCG@10, Recall@100],
    names=["Hybrid (w=20 + RM3)"]
)

,name,AP,R@100,nDCG@10
0,Hybrid (w=20 + RM3),0.271903,0.422951,0.489518


# Experiment 6: Expansion Model Parameter Optimisation

In [42]:
lambdas = [0.3, 0.4, 0.5, 0.6, 0.7]
results = []

for lam in lambdas:
    rm3_lambda = (
        bm25_retr
        >> pt.text.get_text(dataset, ["title", "body"])
        >> pt.apply.text(lambda row: (row['title'] or '') + " " + (row['body'] or ''))
        >> pt.rewrite.RM3(bm25_index_ref, fb_docs=10, fb_terms=10, fb_lambda=lam)
        >> bm25_retr
    )
    
    rm3_final = splade_retr + (20.0 * rm3_lambda)
    
    results.append(
        pt.Experiment(
            [rm3_final],
            topics,
            qrels,
            eval_metrics=[MAP, nDCG@10, Recall@100],
            names=[f"RM3 (lambda={lam})"]
        )
    )

hybrid_results = pd.concat(results)
print(hybrid_results)

               name        AP     R@100   nDCG@10
0  RM3 (lambda=0.3)  0.278107  0.433859  0.480090
0  RM3 (lambda=0.4)  0.277473  0.430455  0.486222
0  RM3 (lambda=0.5)  0.275224  0.427780  0.491013
0  RM3 (lambda=0.6)  0.271903  0.422951  0.489518
0  RM3 (lambda=0.7)  0.267680  0.417757  0.487997
